# 02 — Fold-local Ön İşleme, Manuel / Otomatik HP Araması ve Nested CV
Bu notebook ödevin zorunlu metodolojik iskeletini (Bölüm 5) uygular:

| Bölüm | İçerik |
|---|---|
| A | 6 manuel (M1–M6) ve 6 otomatik (R1–R6) adayın **önceden kaydedilmesi** |
| B | MLP mimarisi, parametre sayısı ve akış şeması |
| C | Nested CV: outer 3 kat × inner 3 kat × 12 aday (108 iç eğitim + 6 dış refit) + referans modeller |
| D | Tam development üzerinde yeniden inner CV (36 eğitim) → nihai yöntem ve konfigürasyon |
| E | Nihai model (3 seed) → **etiketsiz** `test_predictions_locked.csv` |

> Eğitimler CPU'da yapılır. Sonuçlar `results/cache/` altına kaydedildiği için bağlantı koparsa notebook yeniden çalıştırıldığında kaldığı yerden devam eder.

In [ ]:
# === Kurulum: Google Colab (Drive'da ortak klasör) veya yerel Jupyter ===
import os, sys, subprocess
GITHUB_REPO_URL = "https://github.com/ilkinurcan/ysa-mlp-bank-marketing-ilkin.git"
DRIVE_DIR = "/content/drive/MyDrive/YSA_odev"   # üç notebook aynı klasörü kullanır

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    repo_dir = os.path.join(DRIVE_DIR, GITHUB_REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git"))
    if not os.path.exists(os.path.join(repo_dir, "src")):
        subprocess.run(["git", "clone", GITHUB_REPO_URL, repo_dir], check=True)
    else:  # repo'daki kod güncellemelerini al (üretilen sonuç dosyalarına dokunmaz)
        r = subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], capture_output=True, text=True)
        print("git pull:", (r.stdout or r.stderr).strip()[:300])
    os.chdir(repo_dir)
elif not os.path.exists("src"):
    raise RuntimeError("Repo klasöründe değilsin: os.chdir('<repo klasörü>') yap.")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Çalışma klasörü:", os.getcwd())

In [ ]:
import json, time
import numpy as np, pandas as pd
import torch
from IPython.display import display
from src import config, data, plots, search
from src.config import RESULTS_DIR, HIST_DIR
from src.metrics import mean_sd, MAIN_METRICS
from src.model import MLP, count_parameters, parameter_formula

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
seeds = config.get_seeds(); config.set_global_seeds(seeds)
train_cfg = config.training_config()
env = config.write_environment_files()
torch.set_num_threads(max(1, os.cpu_count() or 1))
print("Seeds:", {k: seeds[k] for k in ["base_seed", "torch", "outer_cv", "inner_cv", "random_search", "early_stop_split"]})
print("Torch:", torch.__version__, "| CPU threads:", torch.get_num_threads())
train_cfg

## A. Adayların önceden kaydedilmesi
- **Manuel (M1–M6):** `configs/manual_candidates.json`. Tek-faktör kontrollü tasarım: her aday M1'den yalnızca bir faktörle ayrılır. Ortak varsayılanlar: batch=64, dropout=0, ReLU, WD=1e-4, early stopping açık, clipping yok, `unknown` ayrı kategori.
- **Otomatik (R1–R6):** `ParameterSampler` ve LR için `loguniform`. Optimizer başına tam 2 aday üretilir. SGD için LR aralığı daha geniştir (1e-3–1e-1), çünkü aynı LR'de SGD daha yavaş yakınsar. Adaylar `seed = random_search` ile üretilip `configs/random_candidates.json` dosyasına yazılır. Dosya varsa **yeniden üretilmez**.

In [ ]:
manual = search.load_manual_candidates()
random_ = search.preregister_random_candidates(seeds)
cand_tab = pd.concat([search.candidates_table(manual).assign(search="manual"),
                      search.candidates_table(random_).assign(search="random")], ignore_index=True)
cand_tab.to_csv(RESULTS_DIR / "candidates_preregistered.csv", index=False)
cand_tab

> ✅ **Şimdi commit et:** `configs/random_candidates.json` ve `results/candidates_preregistered.csv` dosyalarını, nested CV sonuçları oluşmadan önce GitHub'a gönder (ör. commit mesajı: *"Pre-register M1–M6 and R1–R6 candidates"*). Bu commit, adayların dış CV sonuçları görülmeden önce sabitlendiğinin kaydıdır.

## B. Veri, MLP mimarisi ve parametre sayısı
Aşağıdaki boyut hesabı yalnızca one-hot sonrası girdi sayısını bulmak içindir; hiçbir seçimde kullanılmaz.

In [ ]:
X_dev, y_dev, g_dev = data.load_development()
print("Development:", X_dev.shape, "| pozitif oranı:", round(y_dev.mean(), 4), "| kopya grubu sayısı:", g_dev.nunique())
from src.preprocessing import preprocessor_for
n_in = preprocessor_for(manual[0]).fit(X_dev).transform(X_dev.iloc[:5]).shape[1]
rows = []
for c in manual + random_:
    parts, total = parameter_formula(n_in, c["hidden_layers"])
    m = MLP(n_in, tuple(c["hidden_layers"]), c["activation"], c["dropout"])
    assert count_parameters(m) == total
    rows.append({"aday": c["id"], "gizli katmanlar": str(tuple(c["hidden_layers"])),
                 "hesap": " + ".join(f"({a}×{b}+{b})" for a, b, _ in parts), "parametre": total})
params_tab = pd.DataFrame(rows); params_tab.to_csv(RESULTS_DIR / "candidate_param_counts.csv", index=False)
print("Ön işleme sonrası girdi boyutu:", n_in)
display(params_tab)
plots.flow_diagram()
plots.mlp_architecture(n_in, manual[0]["hidden_layers"], "ReLU", 0.0, params_tab.iloc[0]["parametre"],
                       name="fig03a_mlp_architecture_M1.png")

## C. Nested CV
Her outer katta:
1. Outer-train içinde 3 inner fold oluşturulur (manuel ve otomatik arama **aynı** inner fold'ları kullanır).
2. Her aday her inner fold'da şöyle eğitilir: inner-train → **fit %85** (ön işleme fit + ağırlık güncelleme) / **ES-monitor %15** (BCE ile early stopping ve LR düşürme) → inner validation'da **AP**.
3. Her arama yönteminde iç CV mean AP'si en yüksek aday (eşitlikte düşük SD) seçilir. Seçilen aday outer-train üzerinde yeniden eğitilir ve outer-evaluation'da **bir kez** ölçülür.
4. Dummy ve Logistic Regression aynı outer-train ile eğitilip aynı outer-evaluation'da ölçülür.
5. Ek analiz: `unknown` kodlama kararı (ayrı kategori / en sık değer) LR üzerinde inner CV ile her outer-train içinde yeniden seçilir.

In [ ]:
t0 = time.time()
res = search.run_nested_cv(X_dev, y_dev, g_dev, manual, random_, seeds, train_cfg)
print(f"Toplam süre: {(time.time() - t0) / 60:.1f} dk")
summ = res["inner_summary"]
summ[summ.search == "manual"].to_csv(RESULTS_DIR / "manual_search_results.csv", index=False)
summ[summ.search == "random"].to_csv(RESULTS_DIR / "random_search_results.csv", index=False)
res["inner_folds"].to_csv(RESULTS_DIR / "inner_cv_fold_results.csv", index=False)
res["outer"].to_csv(RESULTS_DIR / "outer_cv_results.csv", index=False)
res["outer_predictions"].to_csv(RESULTS_DIR / "outer_cv_predictions.csv", index=False)
res["unknown_selection"].to_csv(RESULTS_DIR / "unknown_handling_selection.csv", index=False)
res["split_sizes"].to_csv(RESULTS_DIR / "nested_inner_split_sizes.csv", index=False)
res["outer_split_sizes"].to_csv(RESULTS_DIR / "nested_outer_split_sizes.csv", index=False)
display(res["outer_split_sizes"])

In [ ]:
# Tablo 2 ve 3: aday bazında iç CV mean AP ± SD ve süre (her outer kat için)
cols = ["outer_fold", "candidate", "optimizer", "hidden_layers", "activation", "lr", "weight_decay", "batch_size",
        "scaler", "dropout", "clipping", "scheduler", "inner_AP", "total_time_s", "best_epochs", "warnings", "selected"]
print("MANUEL"); display(summ[summ.search == "manual"][cols])
print("OTOMATİK"); display(summ[summ.search == "random"][cols])

In [ ]:
# Tablo 4: outer kat-bazlı ham sonuçlar
outer = res["outer"]
show = ["outer_fold", "method", "selected_candidate", "inner_AP_of_selected"] + MAIN_METRICS + ["TP", "FP", "TN", "FN", "train_time_s"]
outer[show].round(4)

In [ ]:
# Tablo 5: outer CV Mean ± SD (n=3, ddof=1). Katlar bağımsız tekrar değildir; SD güven aralığı değildir.
outer_summary = mean_sd(outer, "method")
outer_summary.to_csv(RESULTS_DIR / "outer_cv_summary.csv", index=False)
outer_summary[["method"] + MAIN_METRICS + ["n_folds"]]

In [ ]:
# Optimizer davranışı: süre, en iyi epoch, 150 epoch sınırına ulaşma (iç CV eğitimleri)
f = res["inner_folds"].merge(cand_tab[["id", "optimizer"]].rename(columns={"id": "candidate"}), on="candidate")
opt_tab = f.groupby("optimizer").agg(n_egitim=("inner_val_AP", "size"), ort_AP=("inner_val_AP", "mean"),
                                     ort_sure_s=("train_time_s", "mean"), ort_best_epoch=("best_epoch", "mean"),
                                     ort_durulan_epoch=("stopped_epoch", "mean"),
                                     max_epoch_sayisi=("stop_reason", lambda s: int((s == "max_epochs").sum())),
                                     ort_lr_dusurme=("n_lr_reductions", "mean")).round(3)
opt_tab.to_csv(RESULTS_DIR / "optimizer_behaviour_summary.csv")
display(opt_tab)
warn = f[f.warnings.astype(str).str.len() > 0][["context", "candidate", "inner_fold", "stop_reason", "warnings"]]
warn.to_csv(RESULTS_DIR / "training_warnings.csv", index=False)
print(f"Uyarılı eğitim sayısı: {len(warn)} / {len(f)}")

In [ ]:
plots.search_comparison(summ, outer)
hist = {c: pd.read_csv(HIST_DIR / f"outer0__{c}__inner0.csv") for c in ["M1", "M2", "M3", "M6"]}
plots.learning_curves(hist, name="fig04a_learning_curves_M1_M2_M3_M6.png",
                      title="Outer kat 0 / inner kat 0: optimizer ve scheduler etkisi")

## D. Tam development üzerinde yeniden inner CV → nihai seçim
Dış kat sonuçları **yalnızca arama prosedürlerinin genelleme performansını** tanımlar; nihai seçim için kullanılmaz. Nihai yöntem şöyle seçilir: tüm development üzerinde 3-fold inner CV ile iki arama yeniden çalıştırılır (3 × 12 = 36 eğitim). Önceden tanımlı kural uygulanır: **en yüksek mean AP**, eşitlikte düşük SD, o da eşitse manuel.

In [ ]:
dev_folds, dev_summ, final_cfg, decision = search.development_reselection(X_dev, y_dev, g_dev, manual, random_, seeds, train_cfg)
dev_folds.to_csv(RESULTS_DIR / "dev_inner_cv_folds.csv", index=False)
dev_summ.to_csv(RESULTS_DIR / "dev_inner_cv_results.csv", index=False)
config.save_json(decision, RESULTS_DIR / "final_selection_decision.json")
display(dev_summ[["search", "candidate", "optimizer", "hidden_layers", "scaler", "lr", "inner_AP", "total_time_s",
                  "best_epochs", "selected_within_search"]])
decision

## E. Nihai eğitim ve kilitli test tahminleri
- Seçilen konfigürasyon development'tan ayrılan iç early-stop alt kümesiyle (%15) eğitilir.
- **Ana model = birincil seed** (`seeds["primary_seed"]`); seed listesi önceden ilan edilmiştir.
- Aynı konfigürasyon üç ağırlık-başlangıç seed'iyle eğitilir. Model (`.pt`) ve ön işleme durumu (`.joblib`) kaydedilir.
- Logistic Regression ve Dummy tam development üzerinde eğitilir.
- Test olasılıkları ve sınıf tahminleri **etiket sütunu olmadan** `results/test_predictions_locked.csv` dosyasına yazılır; SHA-256 özeti kaydedilir.

In [ ]:
X_test = data.load_test_features()           # y sütunu yok
assert "y" not in X_test.columns
locked, final_info, final_hist, seed_train = search.final_training_and_lock(final_cfg, decision, X_dev, y_dev, g_dev,
                                                                           X_test, seeds, train_cfg)
print("Ana model:", final_info["primary_model"], "| eşik:", final_info["threshold"])
print("SHA-256:", final_info["locked_predictions_sha256"])
display(seed_train[["seed", "is_primary", "best_epoch", "stopped_epoch", "stop_reason", "train_time_s", "n_params", "warnings"]])
display(locked.head())
plots.learning_curves({f"seed {s}": h for s, h in final_hist.items()}, name="fig04b_learning_curves_final.png",
                      title=f"Nihai model ({final_cfg['id']}): eğitim / iç ES kaybı ve LR")
plots.mlp_architecture(final_info["n_inputs_after_preprocessing"], final_cfg["hidden_layers"], final_cfg["activation"],
                       final_cfg["dropout"], final_info["n_trainable_params"], name="fig03b_mlp_architecture_final.png")

> 🛑 **DUR — test etiketlerine bakmadan önce commit et.**
> `results/test_predictions_locked.csv`, `results/test_predictions_locked.sha256`, `results/final_hyperparameters.json`, `results/models/` ve şu ana kadarki tüm kodu **ayrı bir commit** ile GitHub'a gönder (ör. *"Lock final model and unlabeled test predictions"*). 03 notebook'u ancak bundan sonra çalıştır.

In [ ]:
# (Colab) Sonuçları zip olarak indirmek için
import shutil
for d in ["results", "figures", "configs"]:
    shutil.make_archive(f"export_{d}", "zip", root_dir=".", base_dir=d)
print("export_results.zip, export_figures.zip, export_configs.zip oluşturuldu.")
if IN_COLAB:
    from google.colab import files
    for d in ["results", "figures", "configs"]:
        files.download(f"export_{d}.zip")